In [9]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge

In [49]:
X,y = load_diabetes(return_X_y=True)

In [ ]:
class MeraRidgeBGD:
    def __init__(self,epoch=100,learning_rate = 0.01,alpha=0.5):
        self.coef_ = None
        self.intercept_ = None
        self.epoch = epoch
        self.learning_rate = learning_rate
        self.alpha = alpha
    
    def fit(self,X_train,y_train):
        # you need a starting point -> make the coefficients zero
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])
        weights = np.insert(self.coef_ , 0 , self.intercept_)
        X_train = np.insert(X_train,0,1,axis=1)

        for i in range(self.epoch):
            penalty = np.array(self.alpha*weights)
            penalty[0] = 0
            der = (np.dot(X_train.T,X_train).dot(weights) - np.dot(X_train.T,y_train)) / X_train.shape[0] + penalty
            weights = weights - self.learning_rate * der
        
        self.coef_ = weights[1:]
        self.intercept_ = weights[0]
    
    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_ 

__Another Variation with simpler Mathematical Formulae__

In [150]:
class MeraRidgeBGDSimplified:
    def __init__(self,epoch=100,learning_rate=0.01,alpha=0.05):
        self.epoch = epoch
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self,X_train,y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])
        X_train = np.insert(X_train,0,1,axis=1)
        weights = np.insert(self.coef_,0,self.intercept_)
        n = X_train.shape[0]

        for i in range(self.epoch):
            error = np.dot(X_train,weights) - y_train
            penalty = self.alpha * weights
            penalty[0] = 0
            der = np.dot(X_train.T,error) / n + penalty
            weights = weights - self.learning_rate * der
        
        self.intercept_ = weights[0]
        self.coef_ = weights[1:]

    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_


In [141]:

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [123]:
ridge = MeraRidgeBGD(
    500,
    0.1,
    0.1
)

In [111]:
ridge.fit(X_train,y_train)

In [112]:
y_pred = ridge.predict(X_test)

In [113]:
r2_score(y_test,y_pred)

0.05115590719001262

In [151]:
ridge_s = MeraRidgeBGDSimplified(
    500,
    0.1,
    0.1
)

In [152]:
ridge_s.fit(X_train,y_train)

In [153]:
y_pred = ridge_s.predict(X_test)

In [155]:
r2_score(y_test,y_pred)

0.05115590719001262

# Batch Gradient Descent — Formula Reference

---

## Without Regularization

**Expanded Form:**

$$\frac{\partial L}{\partial \beta_j} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i) \cdot X_{ij}$$

**Matrix Form:**

$$\nabla L = \frac{1}{n} X^T(\hat{y} - y)$$

---

## With L2 Regularization (Ridge)

**Expanded Form:**

$$\frac{\partial L}{\partial \beta_j} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i) \cdot X_{ij} + \alpha\beta_j$$

**Matrix Form — Version 1 (clean):**

$$\nabla L = \frac{1}{n} X^T(Xw - y) + \alpha w$$

**Matrix Form — Version 2 (fully expanded):**

$$\nabla L = \frac{1}{n}(X^TXw - X^Ty) + \alpha w$$

---

> **Note:** Both matrix versions are identical — Version 2 is Version 1 with the bracket expanded.  
> Always set `penalty[0] = 0` to avoid regularizing the intercept $\beta_0$.